# Building Your First AI Agent
### A Hands-On Workshop

**What you'll build:** An AI agent that reasons through engineering problems using tools — then use it to optimize a GPU kernel for deep learning.

**What you'll learn:**
- How LLMs become "agents" through tool use
- The agent loop: **reason → act → observe → repeat**
- How to define custom tools and wire them to an LLM
- How agents can experiment with GPU kernel configurations to find the fastest one

**Time:** ~90 minutes | **Prerequisites:** Basic Python

> **Before you begin:** Switch to a GPU runtime: **Runtime → Change runtime type → T4 GPU** (free). This is needed for the kernel optimization sections.

In [ ]:
# Install the Groq SDK (~5 seconds)
!pip install -q groq

## Step 1: Get a Free API Key

1. Go to [Groq Console](https://console.groq.com/keys)
2. Sign up / sign in (no credit card needed)
3. Click **"Create API Key"**
4. Copy the key and paste it in the cell below

The free tier gives you 1,000 requests/day for llama-3.3-70b-versatile model. See https://console.groq.com/settings/limits.

In [ ]:
from groq import Groq
import json
import time

# Paste your API key here
GROQ_API_KEY = "YOUR_API_KEY_HERE"  # <-- Replace this!

client = Groq(api_key=GROQ_API_KEY, max_retries=5)

MODEL = "llama-3.3-70b-versatile"

# Quick test — if this works, you're all set!
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Say 'Hello, engineers!' in a creative way. Keep it short and to one sentence."
        }
    ],
)
print("Connected!", response.choices[0].message.content)

## Step 2: Why Do We Need Agents?

LLMs generate text from patterns in training data. They can reason impressively, but they **can't access your data** — proprietary databases, internal systems, custom materials. Let's see this limitation:

In [ ]:
# Ask the LLM about a proprietary material it has never seen
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "What is the thermal stress in an ExoAlloy X-7 beam heated from 25°C to 300°C?"
        }
    ],
)
print(response.choices[0].message.content)

print("\n" + "="*60)
print("NOTE:\nExoAlloy X-7 is a proprietary material — no LLM has seen it.")
print("The model would either admit ignorance or HALLUCINATE plausible values.")
print("\nThis is the gap that AGENTS fill — by giving LLMs verified TOOLS.")

## Step 3: Build the Tools

We'll give our agent three tools:

| Tool | Purpose |
|---|---|
| `calculate` | Evaluate math expressions precisely |
| `convert_units` | Convert between engineering units |
| `get_material_property` | Look up verified material data |

The LLM **decides when and how** to use these. That's what makes it an **agent**.

> Our material database includes a proprietary alloy (*ExoAlloy X-7*) — data no LLM has seen in training. This simulates real-world tool use: agents accessing company databases, internal APIs, or sensor data.

In [ ]:
import math

# ────────────────────────────────────────────────────────────────
# TOOL 1: Calculator
# ────────────────────────────────────────────────────────────────
def calculate(expression: str) -> str:
    """Evaluates a math expression using Python's math library."""
    try:
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(round(result, 6))
    except Exception as e:
        return f"Error: {e}"


# ────────────────────────────────────────────────────────────────
# TOOL 2: Unit Converter
# ────────────────────────────────────────────────────────────────
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Converts between common engineering units."""
    conversions = {
        ("meters", "feet"): 3.28084,
        ("feet", "meters"): 0.3048,
        ("celsius", "fahrenheit"): lambda v: v * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,
        ("celsius", "kelvin"): lambda v: v + 273.15,
        ("kelvin", "celsius"): lambda v: v - 273.15,
        ("pascals", "psi"): 0.000145038,
        ("psi", "pascals"): 6894.76,
        ("pascals", "mpa"): 1e-6,
        ("mpa", "pascals"): 1e6,
        ("kg", "lbs"): 2.20462,
        ("lbs", "kg"): 0.453592,
        ("watts", "horsepower"): 0.00134102,
        ("horsepower", "watts"): 745.7,
        ("newtons", "pounds_force"): 0.224809,
        ("pounds_force", "newtons"): 4.44822,
        ("meters_per_second", "km_per_hour"): 3.6,
        ("km_per_hour", "meters_per_second"): 1/3.6,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        factor = conversions[key]
        result = factor(value) if callable(factor) else value * factor
        return f"{value} {from_unit} = {result:.6f} {to_unit}"
    available = sorted(set(u for pair in conversions for u in pair))
    return f"Conversion not supported: {from_unit} -> {to_unit}. Available units: {available}"


# ────────────────────────────────────────────────────────────────
# TOOL 3: Material Properties Database
# ────────────────────────────────────────────────────────────────
def get_material_property(material: str, property_name: str) -> str:
    """Looks up a property from a verified engineering materials database."""
    db = {
        "structural_steel": {
            "density": "7850 kg/m^3",  ## Mass per unit volume.
            "youngs_modulus": "200 GPa",  # Stiffness. E = stress / strain. Stress = force / unit area. Strain = change in length / original length.
            "tensile_strength": "400-550 MPa",  # Maximum stress the material can withstand before breaking.
            "yield_strength": "250 MPa",  # Maximum stress the material can withstand before deforming.
            "thermal_expansion_coefficient": "12e-6 per degree C",  # Fractional change in size per degree change in temperature (alpha).
            "thermal_conductivity": "50 W/(m*K)",  # Rate of heat flow per unit area per unit temperature gradient.
            "poissons_ratio": "0.3",  # Deformation in directions perpendicular to the specific direction of loading.
            "melting_point": "1425-1540 C",
                },
        "aluminum_6061": {
            "density": "2700 kg/m^3",
            "youngs_modulus": "68.9 GPa",
            "tensile_strength": "310 MPa",
            "yield_strength": "276 MPa",
            "thermal_expansion_coefficient": "23.6e-6 per degree C",
            "thermal_conductivity": "167 W/(m*K)",
            "poissons_ratio": "0.33",
            "melting_point": "582-652 C",
        },
        "copper": {
            "density": "8960 kg/m^3",
            "youngs_modulus": "117 GPa",
            "tensile_strength": "210-370 MPa",
            "yield_strength": "70 MPa",
            "thermal_expansion_coefficient": "16.5e-6 per degree C",
            "thermal_conductivity": "401 W/(m*K)",
            "poissons_ratio": "0.34",
            "melting_point": "1085 C",
        },
        "titanium_ti6al4v": {
            "density": "4430 kg/m^3",
            "youngs_modulus": "113.8 GPa",
            "tensile_strength": "950 MPa",
            "yield_strength": "880 MPa",
            "thermal_expansion_coefficient": "8.6e-6 per degree C",
            "thermal_conductivity": "6.7 W/(m*K)",
            "poissons_ratio": "0.342",
            "melting_point": "1604-1660 C",
        },
        "concrete_m25": {
            "density": "2400 kg/m^3",
            "youngs_modulus": "25 GPa",
            "compressive_strength": "25 MPa",
            "thermal_expansion_coefficient": "10e-6 per degree C",
            "thermal_conductivity": "1.7 W/(m*K)",
            "poissons_ratio": "0.15-0.20",
        },
        "exoalloy_x7": {
            "density": "4150 kg/m^3",
            "youngs_modulus": "165 GPa",
            "tensile_strength": "820 MPa",
            "yield_strength": "690 MPa",
            "thermal_expansion_coefficient": "9.2e-6 per degree C",
            "thermal_conductivity": "28 W/(m*K)",
            "poissons_ratio": "0.31",
            "melting_point": "1380 C",
        },
    }
    mat = material.lower().replace(" ", "_").replace("-", "_")
    prop = property_name.lower().replace(" ", "_")

    # Fuzzy match the material name
    matched_mat = None
    for key in db:
        if mat in key or key in mat:
            matched_mat = key
            break

    if matched_mat:
        if prop in db[matched_mat]:
            return f"{matched_mat} - {property_name}: {db[matched_mat][prop]}"
        return f"Available properties for {matched_mat}: {', '.join(db[matched_mat].keys())}"
    return f"Material not found. Available: {', '.join(db.keys())}"


# Quick test — make sure the tools work!
print("Calculator:  ", calculate("sqrt(144) + 2 * pi"))
print("Converter:   ", convert_units(100, "celsius", "fahrenheit"))
print("Material DB: ", get_material_property("steel", "youngs_modulus"))
print("Custom DB:   ", get_material_property("exoalloy", "yield_strength"))

In [ ]:
# Now we describe these tools to the LLM using JSON schemas.
# This is the "menu" the model reads to know what's available.

tool_declarations = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": (
                "Evaluates a mathematical expression. Supports basic arithmetic "
                "and math functions: sin, cos, tan, sqrt, log, log10, pi, e, etc."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression to evaluate, e.g. '2 * pi * 0.05' or 'sqrt(3**2 + 4**2)'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_units",
            "description": (
                "Converts a value from one unit to another. Supported units: "
                "meters/feet, celsius/fahrenheit/kelvin, pascals/psi/mpa, kg/lbs, "
                "watts/horsepower, newtons/pounds_force, meters_per_second/km_per_hour."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "The value to convert"},
                    "from_unit": {"type": "string", "description": "Source unit (e.g., 'celsius', 'meters')"},
                    "to_unit": {"type": "string", "description": "Target unit (e.g., 'fahrenheit', 'feet')"}
                },
                "required": ["value", "from_unit", "to_unit"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_material_property",
            "description": (
                "Looks up a physical/mechanical property of an engineering material. "
                "Available materials: structural_steel, aluminum_6061, copper, "
                "titanium_ti6al4v, concrete_m25, exoalloy_x7. "
                "Properties: density, youngs_modulus, tensile_strength, yield_strength, "
                "thermal_expansion_coefficient, thermal_conductivity, poissons_ratio, melting_point."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "material": {"type": "string", "description": "Material name (e.g., 'structural_steel')"},
                    "property_name": {"type": "string", "description": "Property to look up (e.g., 'density')"}
                },
                "required": ["material", "property_name"]
            }
        }
    }
]

print(f"Registered {len(tool_declarations)} tools: {[t['function']['name'] for t in tool_declarations]}")

### Re-try the LLM call giving tools

Let's make one API call with tools and look at the **new response**. We'll ask about the same proprietary material.

In [ ]:
# Ask about a proprietary material no LLM has seen in training
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "What is the yield strength of ExoAlloy X-7?"
        }
    ],
    tools=tool_declarations,
    tool_choice="auto",
)

# Inspect the raw response
message = response.choices[0].message

if message.tool_calls:
    tc = message.tool_calls[0]
    print("The model didn't answer directly — it wants to use a tool!\n")
    print(f"  Tool requested:  {tc.function.name}")
    print(f"  Arguments:       {tc.function.arguments}")
    print(f"\n  KEY INSIGHT: The model has never seen 'ExoAlloy X-7' in training.")
    print(f"  It CHOSE to query our database instead of guessing.")
else:
    print("Model responded with text:", message.content)

## Step 4: The Agent Loop

Now we connect everything. The **agent loop** is the core pattern behind ALL AI agents:

```
User question
     |
     v
+---> LLM reasons about the question
|     |
|     v
|    Does it need a tool? ---NO---> Return final answer
|     |
|    YES
|     |
|     v
|    Execute the tool, get result
|     |
|     v
+--- Feed result back to the LLM
```

This is it. This simple loop powers everything from ChatGPT plugins to autonomous coding agents.

In [ ]:
# Derive the name->function mapping from tool_declarations (one source of truth)
available_tools = {
    tool["function"]["name"]: globals()[tool["function"]["name"]]
    for tool in tool_declarations
}

def execute_tool(name, args):
    """Run a tool by name."""
    if name in available_tools:
        return available_tools[name](**args)
    return f"Unknown tool: {name}"


def run_agent(query, tools=None, system_prompt=None, max_steps=10):
    """
    The Agent Loop: reason -> act -> observe -> repeat.

    Args:
        query: The user's question
        tools: List of tool definitions (uses default if None)
        system_prompt: Optional system instruction
        max_steps: Safety limit to prevent infinite loops
    """
    if tools is None:
        tools = tool_declarations

    print(f"User: {query}\n")

    # Start the conversation
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": query})

    for step in range(max_steps):
        # 1) Call the LLM (retries are handled by the SDK via max_retries)
        # The model occasionally generates a malformed tool call, causing a
        # BadRequestError from the API. This is non-deterministic — re-running
        # the cell usually fixes it.
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                tool_choice="auto",
            )
        except Exception as e:
            print(f"  [API Error] {e}")
            print(f"  Tip: re-run this cell — the model occasionally generates a malformed tool call.")
            return None

        message = response.choices[0].message

        # 2) Check if the model wants to use tools
        #    message.tool_calls is a list, each element has:
        #      .id                  — unique call ID (to match results back)
        #      .function
        #        .name              — which tool (e.g., "calculate")
        #        .arguments         — JSON string (e.g., '{"expression": "2+2"}')
        if message.tool_calls:
            # Add the assistant's message (with tool calls) to history
            messages.append(message)

            for tool_call in message.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                # For code execution, show a short preview instead of the full code
                if name == "run_python_code":
                    code_preview = args.get("code", "")[:80]
                    print(f"  [Tool Call] {name}('{code_preview}...')")
                else:
                    print(f"  [Tool Call] {name}({args})")

                # 3) Execute the tool
                result = execute_tool(name, args)

                # Print result. Show a short additional message for errors.
                print(f"  [Result]    {result}\n")
                if str(result).startswith("Error"):
                    print("Tool call failed — the agent will retry.\n")

                # 4) Feed result back — note: tool_call_id links result to request
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result),
                })
        else:
            # 5) No tool call — model is ready to answer
            if message.content:
                text = message.content
                if len(text) > 300:
                    print(f"  [Thinking] {text[:300]}...\n")
                print(f"Agent: {text}")
            return message.content

    print("Warning: reached maximum steps!")
    return None

print("Agent loop ready! Let's test it.")

In [ ]:
# TEST 1: Material the LLM has never seen — it MUST use our database
run_agent(
    "What is the tensile strength of ExoAlloy X-7? "
    "Can a 20mm diameter rod of this material support a 200kN tensile load without breaking?"
)

In [ ]:
# TEST 2: Multi-step — multiple lookups from our proprietary database + calculation
run_agent(
    "A 5-meter ExoAlloy X-7 beam is fixed at both ends and heated from 25°C to 300°C. "
    "Calculate the thermal stress using stress = E * alpha * delta_T. "
    "Look up the values from our database and check if it exceeds the yield strength."
)

In [ ]:
# TEST 3: Multi-material comparison — proprietary vs standard material
run_agent(
    "Compare the specific strength (yield_strength / density) of ExoAlloy X-7 "
    "and aluminum 6061. Look up all values from our database. "
    "Which is better for a weight-sensitive aerospace application, and by how much?"
)

## Step 5: Level Up — System Prompt + Code Execution

Let's make the agent smarter with two upgrades:
1. **System prompt** — give it an engineering expert persona
2. **Code execution tool** — let it write and run Python for complex tasks

In [ ]:
import io, sys

# ────────────────────────────────────────────────────────────────
# TOOL 4: Python Code Runner
# ────────────────────────────────────────────────────────────────
def run_python_code(code: str) -> str:
    """Executes Python code and returns the printed output."""
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        exec(code, {"__builtins__": __builtins__, "math": math})
        output = sys.stdout.getvalue()
        return output.strip() if output.strip() else "(code executed, no output)"
    except Exception as e:
        return f"Error: {e}"
    finally:
        sys.stdout = old_stdout

# Register it
available_tools["run_python_code"] = run_python_code

# Add the schema (OpenAI format)
code_tool_declaration = {
    "type": "function",
    "function": {
        "name": "run_python_code",
        "description": (
            "Executes Python code and returns printed output. Use for complex calculations, "
            "iterative computations, or anything the other tools can't handle. "
            "Always use print() to show results."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "Python code to execute. Must use print() for output."
                }
            },
            "required": ["code"]
        }
    }
}

# All 4 tools + system prompt for the enhanced agent
all_tool_declarations = tool_declarations + [code_tool_declaration]

SYSTEM_PROMPT = (
    "You are an expert engineering assistant. When solving problems:\n"
    "1. State the approach and relevant formulas first\n"
    "2. Use tools to look up material properties - never guess values\n"
    "3. Use the calculator or code runner for ALL calculations\n"
    "4. Show your work step by step\n"
    "5. Always include units in your final answer"
)

print(f"Enhanced agent ready! Tools: {list(available_tools.keys())}")

In [ ]:
# Test the enhanced agent — database lookup + code execution
run_agent(
    "A cylindrical pressure vessel made from ExoAlloy X-7 has inner radius 0.5m. "
    "Look up its yield strength from the database, then write Python code to calculate "
    "the minimum wall thickness for internal pressures from 10 to 100 MPa (steps of 10). "
    "Use the thin-wall formula: t = (P * r) / (sigma_y / SF), with safety factor SF = 2.0. "
    "Print a table showing pressure vs wall thickness.",
    tools=all_tool_declarations,
    system_prompt=SYSTEM_PROMPT,
)

---

## Step 6: Kernel Optimization — Where Agents Meet Hardware

You've built an agent that uses tools. Now let's point it at a real problem: **GPU kernel optimization in deep learning**.

**Background:** GPU kernels are small programs that run in parallel on thousands of GPU cores. Writing *fast* kernels requires careful tuning — and it's one kind of task AI agents can help with.

**[Triton](https://triton-lang.org/)** is a compiler that lets you write GPU kernels in Python instead of CUDA C. You write a **kernel** — a function that runs on many **programs** in parallel, each processing a **block** of data. Example: a kernel that doubles every element (`x * 2`):

```
 Input:  [ 1  2  3  4 │ 5  6  7  8 │ 9 10 11 12 ]
              │             │             │
              ▼             ▼             ▼
         ┌─────────┐   ┌─────────┐   ┌─────────┐
         │Program 0│   │Program 1│   │Program 2│
         │  x * 2  │   │  x * 2  │   │  x * 2  │
         └────┬────┘   └────┬────┘   └────┬────┘
              │             │             │
              ▼             ▼             ▼
 Output: [ 2  4  6  8 │10 12 14 16 │18 20 22 24 ]

              BLOCK_SIZE = 4,  grid = (3,)
```

Each program gets a unique `program_id` (0–2), loads its block from memory, runs the same kernel code, and stores the result. All 3 programs run **in parallel** on the GPU. *(Figure adapted from [Online Softmax Demystified](https://isztld.com/posts/online-softmax.html))*

**The plan:**
1. Run a simple operation in PyTorch (baseline — two separate GPU operations)
2. Write a Triton kernel that **fuses** it into one GPU pass (fewer memory trips)
3. Ask our agent to **experiment** with different configurations and find the fastest one

In [ ]:
# ──── GPU Setup ────
import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU"
)
print(f"GPU:    {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Triton: {triton.__version__}")


# ──── PYTORCH BASELINE ────
# In eager PyTorch, relu(x + y) runs as TWO separate GPU operations:
#   Kernel 1: temp = x + y    → reads x,y from GPU memory, writes temp back
#   Kernel 2: out = relu(temp) → reads temp from GPU memory, writes out back
# That's 2 round-trips to GPU memory. Can we do it in 1?

def pytorch_add_relu(x, y):
    return torch.relu(x + y)


# ──── NAIVE TRITON KERNEL ────
# Fuse both operations into ONE kernel = ONE memory round-trip.

@triton.jit
def add_relu_kernel(
    x_ptr, y_ptr, output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    """Fused add + ReLU kernel.

    Args:
        x_ptr, y_ptr:  Pointers to input tensors in GPU memory.
        output_ptr:    Pointer to output tensor in GPU memory.
        n_elements:    Total number of elements to process.
        BLOCK_SIZE:    Number of elements each GPU program processes at once.
                       Must be a power of 2. Larger = fewer programs but more
                       work per program. This is the key tuning parameter.
    """
    # Program ID: which block of elements this GPU program is responsible for.
    # Triton launches many programs in parallel — each gets a unique pid.
    pid = tl.program_id(0)

    # Offsets: the global indices this program will process.
    # E.g., pid=0 handles [0..BLOCK_SIZE-1], pid=1 handles [BLOCK_SIZE..2*BLOCK_SIZE-1], etc.
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

    # Mask: the last block may extend past the array end (n_elements may not be
    # divisible by BLOCK_SIZE). The mask prevents out-of-bounds memory access.
    mask = offsets < n_elements

    # Load from GPU memory (one read per input)
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)

    # Fused add + ReLU — no intermediate memory write!
    result = tl.maximum(x + y, 0.0)

    # Store result (one write)
    tl.store(output_ptr + offsets, result, mask=mask)


def triton_add_relu(x, y, BLOCK_SIZE=256):
    """Python wrapper that launches the Triton kernel.

    Args:
        x, y:        Input tensors (same shape, on GPU).
        BLOCK_SIZE:  Elements per program. Determines how many parallel
                     programs the GPU launches (fewer blocks = fewer programs).
    """
    output = torch.empty_like(x)
    n = x.numel()

    # Grid: how many kernel programs to launch in parallel.
    # cdiv = ceiling division: ensures we cover all n elements even if
    # n is not divisible by BLOCK_SIZE (the last program uses a mask).
    # kernel[grid](...) is Triton's launch syntax — like calling a function
    # once per grid element, each running in parallel on the GPU.
    grid = (triton.cdiv(n, BLOCK_SIZE),)
    add_relu_kernel[grid](x, y, output, n, BLOCK_SIZE=BLOCK_SIZE)

    return output


# ──── REUSABLE HELPERS ────

def verify_correctness(kernel_fn, label="Kernel"):
    """Check that kernel_fn(x, y) matches PyTorch's relu(x + y)."""
    x = torch.randn(1_000_000, device='cuda', dtype=torch.float32)
    y = torch.randn(1_000_000, device='cuda', dtype=torch.float32)
    assert torch.allclose(pytorch_add_relu(x, y), kernel_fn(x, y)), f"{label} failed!"
    print(f"{label}: correctness check passed!")


def benchmark_kernels(fns, sizes=[100_000, 1_000_000, 10_000_000, 50_000_000]):
    """Benchmark kernel functions and print a comparison table.

    Args:
        fns:   dict of {"label": fn(x, y)} — each takes two GPU tensors
        sizes: list of input sizes to benchmark
    """
    names = list(fns.keys())
    header = f"{'Size':>12} | " + " | ".join(f"{n:>12}" for n in names)
    print(header)
    print("-" * len(header))
    for s in sizes:
        x = torch.randn(s, device='cuda', dtype=torch.float32)
        y = torch.randn(s, device='cuda', dtype=torch.float32)
        row = f"{s:>12,}"
        for fn in fns.values():
            ms = triton.testing.do_bench(lambda fn=fn: fn(x, y))
            row += f" | {ms:>10.3f}ms"
        print(row)


# ──── VERIFY & BENCHMARK ────
# Triton kernels are JIT-compiled on first launch, which is slow. The
# correctness check doubles as a warmup — after this, benchmark timings
# reflect actual kernel performance, not compilation overhead.
verify_correctness(triton_add_relu, "Naive Triton")

benchmark_kernels({
    "PyTorch": pytorch_add_relu,
    "Triton (256)": triton_add_relu,
})

In [ ]:
# ──── Upgrade the code runner for GPU work ────
# Give it access to torch, triton, and our kernel helpers so the agent
# can experiment with different configurations directly.

import inspect

gpu_namespace = {
    "__builtins__": __builtins__,
    "math": math,
    "torch": torch,
    "triton": triton,
    "tl": tl,
    "pytorch_add_relu": pytorch_add_relu,
    "triton_add_relu": triton_add_relu,
    "verify_correctness": verify_correctness,
    "benchmark_kernels": benchmark_kernels,
}

def run_python_code(code: str) -> str:
    """Executes Python code with access to PyTorch and Triton."""
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        exec(code, gpu_namespace)
        output = sys.stdout.getvalue()
        return output.strip() if output.strip() else "(code executed, no output)"
    except Exception as e:
        import traceback
        return f"Error: {traceback.format_exc()}"
    finally:
        sys.stdout = old_stdout

available_tools["run_python_code"] = run_python_code
print("Code runner upgraded with GPU libraries.")
print("Available: torch, triton, tl, triton_add_relu, verify_correctness, benchmark_kernels")

# ──── Ask the agent to find the best BLOCK_SIZE ────

GPU_SYSTEM_PROMPT = (
    "You are a GPU performance engineer. When optimizing kernels:\n"
    "1. State your approach first\n"
    "2. Always verify correctness before benchmarking\n"
    "3. Use the code runner for ALL experiments\n"
    "4. Report results clearly with a comparison table\n\n"
    "IMPORTANT rules for the code runner:\n"
    "- torch, triton, tl, triton_add_relu, verify_correctness, benchmark_kernels "
    "are ALREADY available — do NOT redefine or re-import them\n"
    "- Always write multi-line Python with newlines (\\n), NEVER use semicolons to "
    "join statements on one line — for loops and if statements require proper indentation"
)

# Pull source from the actual function (one source of truth).
# .fn unwraps the @triton.jit decorator to get the original Python function.
naive_kernel_code = inspect.getsource(add_relu_kernel.fn)

run_agent(
    f"I have this Triton GPU kernel for fused add+ReLU on an NVIDIA T4:\n\n"
    f"```python\n{naive_kernel_code}\n```\n\n"
    f"The wrapper `triton_add_relu(x, y, BLOCK_SIZE=N)` lets you launch it "
    f"with any power-of-2 BLOCK_SIZE. The default is 256.\n\n"
    f"I want to find the fastest BLOCK_SIZE. Use the code runner to run this "
    f"experiment — the helper functions are already available, just call them:\n\n"
    f"```python\n"
    f"fns = {{}}\n"
    f"for bs in [64, 128, 256, 512, 1024, 2048]:\n"
    f"    fn = lambda x, y, bs=bs: triton_add_relu(x, y, BLOCK_SIZE=bs)\n"
    f"    verify_correctness(fn, f'BLOCK_SIZE={{bs}}')\n"
    f"    fns[f'BLOCK_SIZE={{bs}}'] = fn\n"
    f"benchmark_kernels(fns)\n"
    f"```\n\n"
    f"Run the code above, then tell me which BLOCK_SIZE is fastest and why.",
    tools=all_tool_declarations,
    system_prompt=GPU_SYSTEM_PROMPT,
)

### Try it yourself: can the agent write the code on its own?

In the previous cell, we gave the agent a **prescriptive prompt** — the exact Python code to run. The agent's job was just to execute it and interpret the results. That's reliable, but not very "agentic."

What if we just **describe** what we want and let the agent figure out the code itself? That's a **free-form prompt** — closer to how you'd actually talk to an AI assistant.

**Try this:**
1. Run the next cell as-is (free-form prompt, same model as before). Does it work? Does it struggle with multi-line code or closures?
2. Run it again — LLM outputs are **non-deterministic**, so you may see different behavior each time (success, failure, or retries).
3. Change `EXPERIMENT_MODEL` to `"openai/gpt-oss-120b"` and re-run. Does the larger model handle the free-form prompt better?

You'll likely notice: some models need hand-holding (prescriptive prompts), while others can reason through the task on their own. **Run each combination 2–3 times** to get a feel for the variance — one run isn't enough to judge!

Browse available models at: https://console.groq.com/docs/models

In [ ]:
FREE_FORM_PROMPT = (
    f"I have a Triton GPU kernel wrapper `triton_add_relu(x, y, BLOCK_SIZE=N)` "
    f"that does fused add+ReLU. The default BLOCK_SIZE is 256.\n\n"
    f"Write Python code to find the fastest BLOCK_SIZE for 10M elements. "
    f"Try powers of 2 from 64 to 2048. Use `verify_correctness(fn, label)` "
    f"to check each one, and `benchmark_kernels(fns_dict)` to compare them all.\n"
    f"Report which BLOCK_SIZE is fastest."
)

EXPERIMENT_MODEL = "llama-3.3-70b-versatile"  # <-- then try "openai/gpt-oss-120b"

old_model = MODEL
MODEL = EXPERIMENT_MODEL
print(f"Model: {MODEL}")
print(f"{'='*60}\n")

run_agent(
    FREE_FORM_PROMPT,
    tools=all_tool_declarations,
    system_prompt=GPU_SYSTEM_PROMPT,
)

MODEL = old_model  # restore

---

### Beyond Single Agents: The Reflection Pattern

So far we've used a single agent. A powerful extension is **two agents collaborating** — one writes code, another reviews it, and the feedback loops back:

```
┌──────────────┐
│ Coder Agent  │──── writes code ────┐
└──────────────┘                     │
       ▲                             ▼
       │                    ┌─────────────────┐
       │                    │ Reviewer Agent  │
       │                    │ (tests + rates  │
       │                    │  + suggests)    │
       │                    └───────┬─────────┘
       │                            │
       └──── feedback ──────────────┘
              (repeat N rounds)
```

This is the **reflection pattern** — one of the most powerful agentic design patterns. It's how AI coding assistants iteratively improve their own output. You already saw this idea in action: our agent experimented with BLOCK_SIZE values, observed benchmark results, and drew conclusions — a single-agent version of the same loop.

---

### Take-Home Exercise: Softmax Kernel

Softmax is at the heart of every transformer and LLM — it's computed billions of times during inference. Optimizing it has real impact.

**Why is softmax interesting to optimize?**
- It needs **two passes** over the data: once to find the max (numerical stability), once to exponentiate and normalize
- A naive implementation stores intermediate results in GPU memory between passes
- An optimized "online softmax" computes everything in a **single pass** — this is one of the key ideas behind FlashAttention

Below is a working naive softmax kernel. As a take-home exercise:
1. Benchmark it against `torch.softmax`
2. Try different `BLOCK_SIZE` values — which is fastest?
3. Ask the agent about the **online softmax** algorithm — can it be done in a single pass?
4. **Bonus:** Implement the **reflection pattern** from the section above — have a coder agent write an optimized kernel and a reviewer agent critique it, looping until the code improves

**Resource:** [Online Softmax Demystified: From PyTorch to FlashAttention in Triton](https://isztld.com/posts/online-softmax.html) — builds from first principles to working Triton code, step by step.

In [ ]:
# ──────────── TAKE-HOME: Softmax Kernel ────────────

@triton.jit
def softmax_kernel(
    input_ptr, output_ptr,
    n_cols,
    BLOCK_SIZE: tl.constexpr,
):
    """Naive softmax: processes one row per program instance."""
    row_idx = tl.program_id(0)
    row_start = row_idx * n_cols
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_cols

    # Load one row from GPU memory
    row = tl.load(input_ptr + row_start + offsets, mask=mask, other=float('-inf'))

    # Pass 1: find max (for numerical stability)
    row_max = tl.max(row, axis=0)
    row = row - row_max

    # Pass 2: exponentiate and normalize
    numerator = tl.exp(row)
    denominator = tl.sum(numerator, axis=0)
    result = numerator / denominator

    tl.store(output_ptr + row_start + offsets, result, mask=mask)


def triton_softmax(x):
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    output = torch.empty_like(x)
    softmax_kernel[(n_rows,)](x, output, n_cols, BLOCK_SIZE=BLOCK_SIZE)
    return output


# ──── REUSABLE HELPERS (same pattern as add+ReLU) ────

def verify_softmax(fn, label="Softmax"):
    """Check that fn(x) matches torch.softmax(x, dim=-1)."""
    x = torch.randn(1024, 512, device='cuda', dtype=torch.float32)
    assert torch.allclose(torch.softmax(x, dim=-1), fn(x), atol=1e-5), f"{label} failed!"
    print(f"{label}: correctness check passed!")


def benchmark_softmax(fns, shapes=[(1024, 256), (1024, 1024), (4096, 1024), (4096, 4096)]):
    """Benchmark softmax functions across different input shapes."""
    names = list(fns.keys())
    header = f"{'Shape':>16} | " + " | ".join(f"{n:>12}" for n in names)
    print(header)
    print("-" * len(header))
    for rows, cols in shapes:
        x = torch.randn(rows, cols, device='cuda', dtype=torch.float32)
        row = f"{str((rows,cols)):>16}"
        for fn in fns.values():
            ms = triton.testing.do_bench(lambda fn=fn: fn(x))
            row += f" | {ms:>10.3f}ms"
        print(row)


# ──── VERIFY & BENCHMARK ────
# Same warmup pattern: correctness check triggers JIT compilation,
# so subsequent benchmark timings are not skewed by compile time.
verify_softmax(triton_softmax, "Naive Triton")

benchmark_softmax({
    "PyTorch": lambda x: torch.softmax(x, dim=-1),
    "Triton": triton_softmax,
})

print("\nChallenge: Can you close the gap with (or beat) PyTorch's optimized softmax?")
print("Hint: Try different BLOCK_SIZE values, then ask the agent about 'online softmax'.")

# Uncomment to ask the agent for help:
# run_agent(
#     "Here is my naive Triton softmax kernel that processes one row per program:\n\n"
#     "It does two passes: one to find max, one to exponentiate and normalize.\n"
#     "How can I optimize it for an NVIDIA T4? "
#     "Consider: different BLOCK_SIZE values and the online softmax algorithm.",
#     tools=all_tool_declarations,
#     system_prompt=GPU_SYSTEM_PROMPT,
# )

---

## Recap

You built an AI agent from scratch — then used the learnings to optimize GPU kernels. Here's what you learned:

1. **LLMs alone** can generate text but can't interact with the real world — they can't access your databases, run code, or look up proprietary data
2. **Tools** let an LLM interact with the real world — databases, calculators, code runners
3. **The agent loop** (reason → act → observe → repeat) is the core pattern behind all AI agents
4. **The reflection pattern** — two agents collaborating (coder + reviewer) — produces better results than a single agent
5. **Kernel optimization** — fusing operations, tuning block sizes — is what makes AI systems fast enough to be practical, and agents can help do it at scale

### Scaling Up: Model Context Protocol (MCP)

In this workshop, we wired tools to the agent by hand. In production, **[MCP (Model Context Protocol)](https://modelcontextprotocol.io/)** standardizes this — think of it like **USB-C for AI tools**. Any MCP-compatible agent can use any MCP-compatible tool server, without custom integration code.

**How it works:** MCP uses a client-server architecture with three roles:
- **Hosts** — AI applications (like Claude, ChatGPT) that need external context
- **Clients** — Applications that consume MCP server capabilities
- **Servers** — The bridge between AI agents and external systems, exposing **tools**, **resources**, and **prompts**

MCP reduces the integration problem from N×M (every agent × every tool) to **N+M** (each builds one standard interface). As of 2025, MCP has been adopted by OpenAI, Google, and thousands of tool servers. In late 2025, Anthropic donated MCP to the **Agentic AI Foundation** under the Linux Foundation, ensuring it remains an open, vendor-neutral standard.

### Keep Going

**Agentic AI:**
- [Anthropic: Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) — agent design patterns from Anthropic
- [DeepLearning.AI: Agentic AI](https://www.deeplearning.ai/courses/agentic-ai/) — free course by Andrew Ng covering reflection, tool use, planning, and multi-agent patterns
- [Andrew Ng's Four Agentic Design Patterns](https://x.com/AndrewYNg/status/1773393357022298617) — the foundational post on reflection, tool use, planning, multi-agent

**APIs & Tools:**
- [Groq API Docs](https://console.groq.com/docs) — the API we used today (OpenAI-compatible format)
- [Google Gemini API: Function Calling](https://ai.google.dev/gemini-api/docs/function-calling) — alternative API, same concepts

**GPU Kernels:**
- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/) — official kernel writing guide
- [dr2633/triton-kernels](https://github.com/dr2633/triton-kernels) — Colab-ready Triton notebooks
- [Online Softmax Demystified](https://isztld.com/posts/online-softmax.html) — from PyTorch to FlashAttention in Triton

**MCP:**
- [MCP Specification](https://modelcontextprotocol.io/) — official docs and getting started guide
- [Anthropic: Introducing MCP](https://www.anthropic.com/news/model-context-protocol) — why MCP was created
- [MCP on GitHub](https://github.com/modelcontextprotocol/modelcontextprotocol) — specification, SDKs (Python, TypeScript, Java, C#)